In [39]:
import os

import pandas as pd
import rasterio
import geopandas as gpd
import numpy as np
from rasterio import features
from shapely.geometry import Polygon, box
from typing import Tuple
from rasterio.features import rasterize
from rasterio.transform import from_origin


In [ ]:
'''
steps
1. read Central_Africa_samples.csv into a gdf
2. buffer each point by some distance - check the paper if they mention this
3. consider the buffered area as labeled with shifting agriculture
4. divide total gdf bounds into a grid of reasonable size
5. loc examples in each grid cell
6. rasterize and write 

then, use normal appraoch

'''

In [3]:
import sys
sys.path.append('../')
from utils import get_tiles_as_gdf, extract_sample, get_window

In [5]:
df = pd.read_csv('../data/input_datasets/Central_Africa_samples.csv')

In [13]:
df['geometry'] = gpd.points_from_xy(df.X, df.Y, crs="EPSG:4326")
        
gdf = gpd.GeoDataFrame(df)

In [17]:
#Considering the surrounding 0.01 deg (ca. 2km) around each point as areas with smallholder agriculture
gdf['geometry'] = gdf['geometry'].buffer(0.01)

/tmp/ipykernel_682592/1435257698.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['geometry'] = gdf['geometry'].buffer(0.01)


In [18]:
#gdf.to_file('../data/tmp/gdf.gpkg')

In [22]:
bounds = box(*gdf.total_bounds)

In [23]:
print(bounds)

POLYGON ((31.005875000000003 -13.385125, 31.005875000000003 13.040625, 8.553125 13.040625, 8.553125 -13.385125, 31.005875000000003 -13.385125))


In [29]:
def create_grid(square: Polygon, width: float, height: float) -> gpd.GeoDataFrame:
    
    minx, miny, maxx, maxy = square.bounds
    
    ncols = int((maxx - minx) / width)
    nrows = int((maxy - miny) / height)
    
    grid_cells = []
    for i in range(nrows):
        for j in range(ncols):
            # Calculate the coordinates of each cell
            cell_minx = minx + j * width
            cell_miny = miny + i * height
            cell_maxx = cell_minx + width
            cell_maxy = cell_miny + height
            # Create the cell polygon and add it to the list
            grid_cells.append(box(cell_minx, cell_miny, cell_maxx, cell_maxy))
    
    # Create a GeoDataFrame from the grid cells
    grid_gdf = gpd.GeoDataFrame(grid_cells, columns=['geometry'])
    return grid_gdf



In [32]:

cell_width = 10
cell_height = 10
# Create the grid and the GeoDataFrame
grid_gdf = create_grid(bounds, cell_width, cell_height)


In [48]:
def rasterize_gdf(gdf):

    # Determine the bounds of your GeoDataFrame
    minx, miny, maxx, maxy = gdf.total_bounds

    # Set your desired resolution
    resolution = 0.0005 #in degrees

    # Calculate the dimensions of the raster
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)

    # Create the raster's transform
    transform = from_origin(minx, maxy, resolution, resolution)

    # Create the destination raster's profile
    profile = {
        'driver': 'GTiff',
        'height': height,
        'width': width,
        'count': 1,
        'dtype': rasterio.uint8,
        'crs': gdf.crs,
        'transform': transform,
        'nodata': 0
    }

    # Convert the GeoDataFrame's geometry to the format required by rasterize()
    shapes = [(geom, 1) for geom in gdf.geometry]  # Change 1 to another value if needed

    # Rasterize the shapes
    rasterized = rasterize(shapes, out_shape=(height, width), transform=transform,
                           fill=0, all_touched=True, dtype='uint8')
    return rasterized, profile



In [50]:
for idx, row in grid_gdf.iterrows():
    print(idx)
    grid_items = gdf.loc[gdf.intersects(row.geometry)]

    array, profile = rasterize_gdf(grid_items)
        
    ## Save the rasterized shapes to a new file
    with rasterio.open(f"../data/input_datasets/tyukavina_congo_basin_agri/{idx}.tif", 'w', **profile) as dst:
        dst.write(array, 1)

0
1
2
3


In [ ]:
print('asfd')

In [3]:
lc_tiles_dir = '/home/jan/Documents/EPFL/project_2/data/input_datasets/worldcover'
lc_tiles = get_tiles_as_gdf(lc_tiles_dir)

cocoa_ecuador = gpd.read_file('../data/tmp/cacao_ecuador.gpkg')
cocoa_ecuador = cocoa_ecuador.to_crs(4326)

In [4]:
dataset_extent = box(*cocoa_ecuador.total_bounds)

In [5]:
lc_tiles_filtered = lc_tiles.loc[lc_tiles.intersects(dataset_extent)]

In [8]:
out_dir = '../data/input_datasets/cocoa_ecuador'
for idx, row in lc_tiles_filtered.iterrows():
    print(idx)
    cocoa_ecuador_in_tile = cocoa_ecuador.loc[cocoa_ecuador.intersects(row.geometry)]
    
    with rasterio.open(row.filename) as src:
        lc_map = src.read(1)
        profile = src.profile

        shapes = [(geom, 1) for geom in cocoa_ecuador_in_tile.geometry]
        if not shapes:
            continue
        # Rasterize the shapes
        rasterized_cocoa = features.rasterize(shapes, out_shape=(profile['height'], profile['width']),
                               transform=profile['transform'], fill=0, all_touched=True, dtype='uint8')

    # Conditions
    is_forest_cover = lc_map == 10
    is_cocoa = rasterized_cocoa == 1
    cond = is_forest_cover & is_cocoa
    # New array based on conditions
    coco_filtered = np.where(cond, 1, 0) 

    out_filename = os.path.join(out_dir, row.filename.split('/')[-1].split('_')[-2] + '.tif')
    
    #Save the rasterized shapes to a new file
    with rasterio.open(out_filename, 'w', **profile) as dst:
        dst.write(coco_filtered, 1)

432
433
618
619
638
639
